# Visium HD Spatial Transcriptomics Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stevenpastor/spatial_transcriptomics_resources/blob/main/notebook/visium_hd_quickstart.ipynb)

A condensed walkthrough of Visium HD analysis, run on the **10x Genomics Human CRC Patient 1** dataset ([Nature Genetics 2025](https://www.nature.com/articles/s41588-025-02193-3)).

---

## Setup

Install the required packages and pull the pre-processed data from Figshare. The download includes a binned count matrix, a pre-computed annotated object, ground-truth labels, and an H&E tissue image.

In [ ]:
# Package installation + data download
import os, sys, subprocess
from pathlib import Path

# Install packages
print("Installing packages...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "scanpy>=1.10", "squidpy>=1.6", "anndata>=0.10",
    "numpy>=1.24", "pandas>=2.0", "scipy>=1.11",
    "matplotlib>=3.7", "seaborn>=0.13",
    "scikit-image>=0.21", "Pillow>=10",
    "igraph", "leidenalg",
    "tqdm>=4.65", "requests>=2.31", "h5py>=3.9", "pyarrow>=13",
])
print("Done.")

# Download precomputed data from Figshare. The main h5ads + spatial images
# live in one article; the LIANA results parquet is published separately.
FIGSHARE_ARTICLE_IDS = ["31937262", "32273760"]

DATA_DIR = Path("/content/data")
PRECOMPUTED_DIR = DATA_DIR / "precomputed"
SPATIAL_DIR = PRECOMPUTED_DIR / "spatial"
PRECOMPUTED_DIR.mkdir(parents=True, exist_ok=True)
SPATIAL_DIR.mkdir(parents=True, exist_ok=True)

RAW_H5AD = PRECOMPUTED_DIR / "crc_p1_raw_50k.h5ad"
ANNOT_H5AD = PRECOMPUTED_DIR / "crc_p1_annotated_50k.h5ad"
LIANA_PARQUET = PRECOMPUTED_DIR / "crc_p1_liana_full.parquet"

SPATIAL_FILENAMES = {
    "tissue_hires_image.png",
    "tissue_lowres_image.png",
    "scalefactors_json.json",
}

def _target_path(name):
    return SPATIAL_DIR / name if name in SPATIAL_FILENAMES else PRECOMPUTED_DIR / name

def download_file(url, dest, desc="Downloading"):
    import requests
    from tqdm.auto import tqdm
    tmp = dest.with_suffix(dest.suffix + ".part")
    with requests.get(url, stream=True, timeout=(60, None)) as resp:
        resp.raise_for_status()
        total = int(resp.headers.get("content-length", 0))
        with open(tmp, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=desc) as pbar:
            for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    tmp.replace(dest)
    print(f"Saved: {dest.relative_to(DATA_DIR)} ({dest.stat().st_size / 1e6:.2f} MB)")

def fetch_figshare_files():
    import requests
    entries = []
    for aid in FIGSHARE_ARTICLE_IDS:
        resp = requests.get(f"https://api.figshare.com/v2/articles/{aid}/files", timeout=60)
        resp.raise_for_status()
        entries.extend(resp.json())
    return entries

needed = [
    RAW_H5AD,
    ANNOT_H5AD,
    LIANA_PARQUET,
    SPATIAL_DIR / "tissue_hires_image.png",
    SPATIAL_DIR / "tissue_lowres_image.png",
    SPATIAL_DIR / "scalefactors_json.json",
]

if not all(p.exists() for p in needed):
    print("Downloading precomputed data from Figshare (per file)...")
    for entry in fetch_figshare_files():
        name = entry["name"]
        dest = _target_path(name)
        expected_size = entry.get("size")
        if dest.exists() and expected_size is not None and dest.stat().st_size == expected_size:
            print(f"Skipping (already present): {dest.relative_to(DATA_DIR)}")
            continue
        download_file(entry["download_url"], dest, desc=name)
    print("Download complete.")
else:
    print("Precomputed data already downloaded.")

# Clone repo for utils.py
REPO_URL = "https://github.com/stevenpastor/spatial_transcriptomics_resources.git"
REPO_DIR = Path("/content/spatial_tutorial")
if not REPO_DIR.exists():
    print("Cloning tutorial repo (for utility functions)...")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
SCRIPTS_DIR = REPO_DIR / "scripts"
print("Ready.")

In [ ]:
import sys, os, time, warnings, gc
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False, fontsize=12)

# Paths
DATA_DIR = Path("/content/data")
PRECOMPUTED_DIR = DATA_DIR / "precomputed"
FIG_DIR = Path("/content/figures")
FIG_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(SCRIPTS_DIR))
from utils import (
    load_visium_hd_binned,
    compute_qc_metrics,
    spatial_qc_plots,
    spatial_outlier_detection,
)

print(f"scanpy {sc.__version__}, squidpy {sq.__version__}")

## Data Loading

What we load here is a random subsample of the full ~500K+ 8 um binned CRC Patient 1 dataset. The subsample was created by `scripts/generate_precomputed.py` to keep file sizes under 100 MB for Colab. Filenames say "50k" (the target), but the actual bin count can drift a little because of filtering.

In [ ]:
%%time
# List precomputed files
for f in sorted(PRECOMPUTED_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6 if f.is_file() else 0
    print(f"  {f.name:40s}  {size_mb:6.1f} MB" if f.is_file() else f"  {f.name}/")

# Load 8 um binned data (random subsample from the full dataset)
adata_8um = sc.read_h5ad(PRECOMPUTED_DIR / "crc_p1_raw_50k.h5ad")
print(f"\nLoaded: {adata_8um.shape[0]:,} bins x {adata_8um.shape[1]:,} genes")
print(f"Spatial range X: [{adata_8um.obsm['spatial'][:,0].min():.0f}, {adata_8um.obsm['spatial'][:,0].max():.0f}]")
print(f"Spatial range Y: [{adata_8um.obsm['spatial'][:,1].min():.0f}, {adata_8um.obsm['spatial'][:,1].max():.0f}]")
if sparse.issparse(adata_8um.X):
    print(f"Median UMIs/bin: {np.median(adata_8um.X.sum(axis=1).A1):.0f}")
    print(f"Median genes/bin: {np.median((adata_8um.X > 0).sum(axis=1).A1):.0f}")

# Ground-truth annotations (from the original study group, not us)
if "UnsupervisedL1" in adata_8um.obs.columns:
    print(f"\nGround-truth annotations present: {adata_8um.obs['UnsupervisedL1'].notna().sum():,} / {adata_8um.shape[0]:,} bins")
    print(adata_8um.obs["UnsupervisedL1"].value_counts().to_string())

## H&E Image Verification

Overlay the bin positions on the H&E to confirm that spatial coordinates are registered to the histology correctly.

In [ ]:
%%time
# Load H&E and overlay bin positions
import json as _json

if "spatial_image" in adata_8um.uns:
    he_image = adata_8um.uns["spatial_image"]
    scalefactors = adata_8um.uns.get("scalefactors", {})
    # Embedded image is lowres, so use the lowres scale factor
    sf = scalefactors.get("tissue_lowres_scalef", scalefactors.get("tissue_hires_scalef", 1.0))
    print(f"Image from adata.uns, shape={he_image.shape}, sf={sf}")
else:
    from skimage import io as skio
    spatial_dir = PRECOMPUTED_DIR / "spatial"
    he_image = None
    for name in ["tissue_hires_image.png", "tissue_lowres_image.png"]:
        candidate = spatial_dir / name
        if candidate.exists():
            he_image = skio.imread(str(candidate))
            with open(spatial_dir / "scalefactors_json.json") as f:
                scalefactors = _json.load(f)
            # Match scale factor to whichever image was loaded
            key = "tissue_hires_scalef" if "hires" in name else "tissue_lowres_scalef"
            sf = scalefactors.get(key, 1.0)
            break

if he_image is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Left: H&E only
    axes[0].imshow(he_image)
    axes[0].set_title("H&E Image")
    axes[0].axis("off")

    # Right: bins overlaid on semi-transparent H&E
    coords = adata_8um.obsm["spatial"]
    axes[1].imshow(he_image, alpha=0.3)
    axes[1].scatter(
        coords[:, 0] * sf, coords[:, 1] * sf,
        s=0.3, c="red", alpha=0.5, rasterized=True,
    )
    # Lock axes to the image extent so the H&E isn't shrunk to an inset
    axes[1].set_xlim(0, he_image.shape[1])
    axes[1].set_ylim(he_image.shape[0], 0)
    axes[1].set_title("Bins overlaid on H&E (semi-transparent)")
    axes[1].axis("off")

    plt.tight_layout()
    plt.savefig(FIG_DIR / "crc_image_verification.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Bins plotted: {coords.shape[0]:,}")
else:
    print("H&E image not found. Plotting spatial coordinates only.")
    fig, ax = plt.subplots(figsize=(8, 7))
    coords = adata_8um.obsm["spatial"]
    total_counts = np.asarray(adata_8um.X.sum(axis=1)).flatten() if sparse.issparse(adata_8um.X) else adata_8um.X.sum(axis=1)
    sc_plot = ax.scatter(coords[:, 0], coords[:, 1], s=0.1,
                         c=total_counts, cmap="viridis", edgecolors="none", rasterized=True)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
    ax.set_title("Bin positions (colored by UMI count)")
    plt.colorbar(sc_plot, ax=ax, shrink=0.6)
    plt.tight_layout()
    plt.show()

### What this means

* The left panel is the H&E histology image. On the right, all bin positions show as red dots over a semi-transparent copy of the same image. The dots should trace the tissue boundary cleanly, with no offset, rotation, or mirroring.

* The dots look sparse because this is a random subsample of ~500K+ total bins. Any gaps in coverage are from subsampling, not lost data.

## Per-Bin Quality Metrics

First, compute standard QC metrics for each bin and look at their distributions. This is the same kind of QC you would run for scRNA-seq, just applied to spatial bins.

| Metric | What it measures | High values suggest |
|--------|-----------------|-------------------|
| `total_counts` | Total UMIs per bin | Dense tissue / doublets |
| `n_genes_by_counts` | Genes detected per bin | Transcriptomic complexity |
| `pct_counts_mt` | % mitochondrial UMIs | Dying/stressed cells |
| `complexity` | log(genes)/log(counts) | Transcriptomic diversity |

In [ ]:
%%time
# Compute QC metrics
adata_8um = compute_qc_metrics(adata_8um)

print("QC metric summary:")
for col in ["total_counts", "n_genes_by_counts", "pct_counts_mt", "complexity"]:
    vals = adata_8um.obs[col]
    print(f"  {col:25s}  median={vals.median():8.1f}  mean={vals.mean():8.1f}  "
          f"[{vals.min():.1f}, {vals.max():.1f}]")

# Violin plots
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, metric in zip(axes, ["total_counts", "n_genes_by_counts",
                              "pct_counts_mt", "complexity"]):
    parts = ax.violinplot(adata_8um.obs[metric].dropna().values, showmedians=True, showextrema=False)
    for pc in parts["bodies"]:
        pc.set_facecolor("steelblue"); pc.set_alpha(0.7)
    ax.set_title(metric.replace("_", "\n"), fontsize=10)
    ax.set_xticks([])
plt.suptitle("QC Distributions (8 um bins)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_qc_violins.png", dpi=150, bbox_inches="tight")
plt.show()

## Scatter: genes vs UMIs colored by mt%
#fig, ax = plt.subplots(figsize=(8, 6))
#scatter = ax.scatter(
#    adata_8um.obs["total_counts"], adata_8um.obs["n_genes_by_counts"],
#    c=adata_8um.obs["pct_counts_mt"], s=0.5, cmap="RdYlBu_r",
#    edgecolors="none", rasterized=True,
#    vmin=0, vmax=adata_8um.obs["pct_counts_mt"].quantile(0.95),
#)
#ax.set_xlabel("Total UMI counts"); ax.set_ylabel("Genes detected")
#ax.set_title("Gene complexity colored by mitochondrial %")
#plt.colorbar(scatter, ax=ax, label="pct_counts_mt")
#plt.tight_layout()
#plt.savefig(FIG_DIR / "crc_qc_scatter.png", dpi=150, bbox_inches="tight")
#plt.show()

# Histograms with data-driven thresholds
min_counts = max(adata_8um.obs["total_counts"].quantile(0.01), 10)
max_counts = adata_8um.obs["total_counts"].quantile(0.995)
min_genes = max(adata_8um.obs["n_genes_by_counts"].quantile(0.01), 5)
max_mt = min(adata_8um.obs["pct_counts_mt"].quantile(0.95), 25)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
thresholds = [
    ("total_counts", min_counts, max_counts, "UMI Counts"),
    ("n_genes_by_counts", min_genes, None, "Genes Detected"),
    ("pct_counts_mt", None, max_mt, "Mitochondrial %"),
    ("complexity", None, None, "Complexity"),
]
for ax, (metric, lo, hi, label) in zip(axes, thresholds):
    ax.hist(adata_8um.obs[metric].dropna(), bins=100, color="steelblue", alpha=0.7, edgecolor="none")
    if lo is not None:
        ax.axvline(lo, color="red", ls="--", lw=1.5, label=f"min={lo:.1f}")
    if hi is not None:
        ax.axvline(hi, color="red", ls="--", lw=1.5, label=f"max={hi:.1f}")
    ax.set_title(label)
    if lo is not None or hi is not None:
        ax.legend(fontsize=8)
plt.suptitle("QC Histograms with Suggested Thresholds", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_qc_histograms.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Suggested thresholds: min_counts={min_counts:.1f}, max_counts={max_counts:.1f}, "
      f"min_genes={min_genes:.1f}, max_mt={max_mt:.1f}%")

### What this means

* **UMIs/bin**: Median ~147. That is what we expect at 8 um resolution, where each bin covers a tiny patch of tissue. The right-skewed tail reflects bins sitting over dense or transcriptionally active regions.

* **Mitochondrial %**: Median 6.7%, but the tail goes all the way to 100%. Bins at 100% mt are either dead/damaged cells or ambient mitochondrial RNA. Why does mt% run high in tumor cells? Tumor cells are metabolically hyperactive and often grow under hypoxia, both of which push mitochondrial gene expression up. So a chunk of the elevated mt% in tumor regions is real biology rather than damage. The trick is to check it spatially. High-mt bins clustered at tissue edges and folds are damage and should be filtered. High-mt bins dispersed through the tumor core are more likely real signal.

* These global metrics catch problematic bins, but they treat every bin the same way regardless of where it sits. Next we layer in spatial QC to catch the location-dependent artifacts.

## Spatial QC

### Bins outside tissue

The Visium HD capture area is a 6.5 x 6.5 mm square, but the tissue section rarely fills the whole thing. Space Ranger flags each bin as `in_tissue` (1) or off-tissue (0) based on whether it sits under the section. Off-tissue bins only capture ambient RNA and background noise, so they have to come out before analysis. Our loading function (`load_visium_hd_binned` in `utils.py`) already restricts to `in_tissue == 1`, but some edge bins that technically sit under tissue still produce very low signal because they only partly overlap the section. Those are what we visualize next.

### Why spatial QC on top of global QC?

Global thresholds treat every bin identically. 50 UMIs is suspicious in dense tumor but normal in sparse stroma. Spatial QC instead compares each bin to its neighbors and flags the ones that drift from their local context. That catches tissue folds, air bubbles, and edge artifacts that look fine in a histogram but cluster in space.

In [ ]:
%%time
# --- Tissue edge bins ---
coords = adata_8um.obsm["spatial"]

if "in_tissue" in adata_8um.obs.columns:
    print(f"in_tissue values: {adata_8um.obs['in_tissue'].value_counts().to_dict()}")
    print("All bins are in_tissue == 1 (off-tissue bins were removed during loading)")
else:
    print("in_tissue column was dropped after filtering (all remaining bins are in-tissue)")

print(f"Bins in dataset: {adata_8um.shape[0]:,}")

total_cts = np.asarray(adata_8um.X.sum(axis=1)).flatten() if sparse.issparse(adata_8um.X) else adata_8um.X.sum(axis=1)
low_count_thresh = np.percentile(total_cts, 5)
is_edge = total_cts < low_count_thresh

sq.gr.spatial_neighbors(adata_8um, n_neighs=20, coord_type="generic")
edge_outlier = np.zeros(adata_8um.shape[0], dtype=bool)
for _metric in ["total_counts", "pct_counts_mt", "n_genes_by_counts"]:
    edge_outlier = edge_outlier | spatial_outlier_detection(
        adata_8um, metric=_metric, z_threshold=3.0).values
is_edge = is_edge & edge_outlier

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: all in-tissue bins colored by UMI count
sc_plot = axes[0].scatter(
    coords[:, 0], coords[:, 1], s=0.2, c=total_cts,
    cmap="viridis", edgecolors="none", rasterized=True,
)
axes[0].set_aspect("equal"); axes[0].invert_yaxis(); axes[0].axis("off")
axes[0].set_title("In-tissue bins (colored by UMI count)")
plt.colorbar(sc_plot, ax=axes[0], shrink=0.6, label="Total UMIs")

# Right: tissue edge bins removed in QC
axes[1].scatter(
    coords[~is_edge, 0], coords[~is_edge, 1], s=0.2,
    c="steelblue", alpha=0.3, edgecolors="none", rasterized=True, label="Other bins",
)
axes[1].scatter(
    coords[is_edge, 0], coords[is_edge, 1], s=0.5,
    c="red", alpha=0.6, edgecolors="none", rasterized=True,
    label="Edge bins removed",
)
axes[1].set_aspect("equal"); axes[1].invert_yaxis(); axes[1].axis("off")
axes[1].set_title("Tissue edge bins removed in QC")
axes[1].legend(markerscale=10, fontsize=10, loc="lower right")

plt.tight_layout()
plt.savefig(FIG_DIR / "crc_tissue_boundary.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Edge bins removed: {is_edge.sum():,} / {len(is_edge):,}")

In [ ]:
%%time
# --- Spatial QC heatmaps ---
spatial_qc_plots(
    adata_8um,
    metrics=["total_counts", "n_genes_by_counts", "pct_counts_mt", "complexity"],
    figsize=(22, 4), spot_size=0.3,
    save=str(FIG_DIR / "crc_spatial_qc_heatmaps.png"),
)

In [ ]:
%%time
# --- Spatial outlier detection ---
print("Computing spatial neighbors (k=20)...")
sq.gr.spatial_neighbors(adata_8um, n_neighs=20, coord_type="generic")

outlier_metrics = ["total_counts", "pct_counts_mt", "n_genes_by_counts"]
total_outliers = np.zeros(adata_8um.shape[0], dtype=bool)
for metric in outlier_metrics:
    outliers = spatial_outlier_detection(adata_8um, metric=metric, z_threshold=3.0)
    n_out = outliers.sum()
    print(f"  {metric:25s}: {n_out:,} outliers ({100*n_out/len(outliers):.2f}%)")
    total_outliers = total_outliers | outliers.values

adata_8um.obs["spatial_outlier"] = total_outliers
print(f"\nTotal spatial outliers: {total_outliers.sum():,} ({100*total_outliers.mean():.2f}%)")

fig, ax = plt.subplots(figsize=(8, 7))
coords = adata_8um.obsm["spatial"]
ax.scatter(coords[~total_outliers, 0], coords[~total_outliers, 1],
           s=0.1, c="lightgray", alpha=0.3, rasterized=True, label="Normal")
ax.scatter(coords[total_outliers, 0], coords[total_outliers, 1],
           s=1, c="red", alpha=0.6, rasterized=True, label="Outlier")
ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
ax.legend(markerscale=10, fontsize=10)
ax.set_title(f"Spatial outliers (n={total_outliers.sum():,})")
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_spatial_outliers.png", dpi=150, bbox_inches="tight")
plt.show()

### What this means

* **Tissue edge bins**: The left panel shows UMI density across the tissue footprint. Higher counts (yellow) line up with dense tissue regions, lower counts (purple) with sparse stroma or tissue edges. The right panel shows the low-UMI bins along the tissue boundary that are removed in QC, where bins only partially overlap the section.

* **Spatial heatmaps**: Total counts and genes both show a gradient, with higher signal in the dense tumor region and lower signal in the stroma. Mitochondrial % is diffuse rather than piling up at the edges, which suggests a mix of biology and tissue quality.

* **Spatial outlier detection**: For every bin, we compute a z-score of three QC metrics (`total_counts`, `pct_counts_mt`, `n_genes_by_counts`) against its 20 nearest spatial neighbors and flag any bin where `|z| > 3` on at least one metric. About 3-4% of bins land as outliers, and they are scattered across the tissue rather than clumping in one region. That is reassuring: there's no large artifact zone (tissue fold, air bubble) distorting the data. These bins are removed in QC.

## Filtering and Normalization

The QC filter applied below combines five criteria into a single `keep` mask:

- `total_counts >= min_counts` (1st percentile of `total_counts`, or 10, whichever is larger)
- `total_counts <= max_counts` (99.5th percentile)
- `n_genes_by_counts >= min_genes` (1st percentile, or 5)
- `pct_counts_mt <= max_mt` (95th percentile, capped at 25)
- `~spatial_outlier` (the union of the three spatial-outlier flags from the previous step)

After filtering, we load a precomputed object that has already been normalized, dimensionally reduced, and clustered. The commented-out code below is what those steps look like if you wanted to run them yourself.

In [ ]:
%%time
# Apply global + spatial QC filters
n_before = adata_8um.shape[0]

min_counts = max(adata_8um.obs["total_counts"].quantile(0.01), 10)
max_counts = adata_8um.obs["total_counts"].quantile(0.995)
min_genes = max(adata_8um.obs["n_genes_by_counts"].quantile(0.01), 5)
max_mt = min(adata_8um.obs["pct_counts_mt"].quantile(0.95), 25)

keep = (
    (adata_8um.obs["total_counts"] >= min_counts) &
    (adata_8um.obs["total_counts"] <= max_counts) &
    (adata_8um.obs["n_genes_by_counts"] >= min_genes) &
    (adata_8um.obs["pct_counts_mt"] <= max_mt) &
    (~adata_8um.obs["spatial_outlier"].astype(bool))
)
adata_8um = adata_8um[keep].copy()
min_cells = max(int(0.001 * adata_8um.shape[0]), 3)
sc.pp.filter_genes(adata_8um, min_cells=min_cells)

print(f"Filtering: {n_before:,} -> {adata_8um.shape[0]:,} bins "
      f"(removed {n_before - adata_8um.shape[0]:,}, "
      f"{100*(n_before - adata_8um.shape[0])/n_before:.1f}%)")

# Load precomputed normalized + clustered data.
# I ran normalization, PCA, UMAP, and Leiden clustering separately and saved
# the result. The bin count differs slightly from the filtering above because
# the precomputed file was generated from a separate run. Here is what those
# steps would look like:
#
# sc.pp.normalize_total(adata_8um, target_sum=1e4)
# sc.pp.log1p(adata_8um)
# sc.pp.highly_variable_genes(adata_8um, n_top_genes=2000, flavor="seurat_v3")
# adata_8um = adata_8um[:, adata_8um.var["highly_variable"]].copy()
# sc.pp.scale(adata_8um, max_value=10)
# sc.tl.pca(adata_8um, n_comps=50)
# sc.pp.neighbors(adata_8um, n_neighbors=15, n_pcs=30)
# sc.tl.umap(adata_8um)
# sc.tl.leiden(adata_8um, resolution=0.5)

adata_8um = sc.read_h5ad(PRECOMPUTED_DIR / "crc_p1_annotated_50k.h5ad")
print(f"Loaded precomputed annotated data: {adata_8um.shape}")
print(f"Leiden clusters: {adata_8um.obs['leiden'].nunique()}")

# UMAP colored by QC metrics + Leiden
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, metric, cmap in zip(axes[:3],
    ["total_counts", "n_genes_by_counts", "pct_counts_mt"],
    ["viridis", "viridis", "RdYlBu_r"]):
    sc_p = ax.scatter(adata_8um.obsm["X_umap"][:, 0], adata_8um.obsm["X_umap"][:, 1],
                      c=adata_8um.obs[metric], s=0.3, cmap=cmap,
                      edgecolors="none", rasterized=True)
    ax.set_title(metric); plt.colorbar(sc_p, ax=ax, shrink=0.7)

for cl in sorted(adata_8um.obs["leiden"].unique(), key=int):
    mask = adata_8um.obs["leiden"] == cl
    axes[3].scatter(adata_8um.obsm["X_umap"][mask, 0], adata_8um.obsm["X_umap"][mask, 1],
                    s=0.3, label=cl, alpha=0.5, rasterized=True)
axes[3].set_title("Leiden clusters")
axes[3].legend(markerscale=5, fontsize=7, ncol=2, loc="best")
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_umap_qc_leiden.png", dpi=150, bbox_inches="tight")
plt.show()

### What this means

* Combined spatial + global filtering took out ~22% of bins. After filtering, the UMAP shows clear cluster structure.

* High-count/high-mt bins cluster together in the bottom-right. That is probably biological (metabolically active tumor epithelium) rather than pure artifact, since the same region maps to tumor cell types when we check annotations next.

## Ground-Truth Cell Type Annotations

These annotations come from the **original study group** ([de Oliveira et al., Nature Genetics 2025](https://www.nature.com/articles/s41588-025-02193-3)), not from us. They used unsupervised clustering and deconvolution on the full dataset. We use their labels as the reference for evaluating our own annotation approach.

**UnsupervisedL1** gives 10 broad categories: Tumor, Fibroblast, T cells, B cells, Myeloid, Endothelial, Smooth Muscle, Intestinal Epithelial, Neuronal, Unknown.

In [ ]:
# Visualize ground-truth UnsupervisedL1
ct_col = "UnsupervisedL1"
has_label = adata_8um.obs[ct_col].notna()
adata_labeled = adata_8um[has_label].copy()
print(f"Bins with {ct_col}: {has_label.sum():,} / {adata_8um.shape[0]:,}")

cell_types = sorted(adata_labeled.obs[ct_col].dropna().unique())

# Shared palette spanning ground-truth labels plus the marker-based categories
# defined later, so both plots use identical colors per cell type.
all_cell_types = sorted(set(cell_types) | {
    "Tumor", "Fibroblast", "Endothelial", "Myeloid", "T cells", "B cells",
    "Smooth Muscle", "Intestinal Epithelial", "Low_confidence",
})
cell_type_palette = dict(zip(all_cell_types, plt.cm.tab20(np.linspace(0, 1, len(all_cell_types)))))
palette = cell_type_palette

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ct in cell_types:
    mask = (adata_labeled.obs[ct_col] == ct).values
    axes[0].scatter(adata_labeled.obsm["X_umap"][mask, 0], adata_labeled.obsm["X_umap"][mask, 1],
                    s=0.5, label=ct, alpha=0.5, rasterized=True, color=palette[ct])
axes[0].legend(markerscale=8, fontsize=8, loc="best", framealpha=0.8)
axes[0].set_title(f"UMAP: {ct_col} (ground truth)"); axes[0].set_xlabel("UMAP 1"); axes[0].set_ylabel("UMAP 2")

coords = adata_labeled.obsm["spatial"]
for ct in cell_types:
    mask = (adata_labeled.obs[ct_col] == ct).values
    axes[1].scatter(coords[mask, 0], coords[mask, 1],
                    s=0.3, label=ct, alpha=0.4, rasterized=True, color=palette[ct])
axes[1].legend(markerscale=10, fontsize=8, loc="best", framealpha=0.8)
axes[1].set_title(f"Spatial: {ct_col}"); axes[1].set_aspect("equal"); axes[1].invert_yaxis(); axes[1].axis("off")

plt.tight_layout()
plt.savefig(FIG_DIR / "crc_ground_truth_unsupervised.png", dpi=150, bbox_inches="tight")
plt.show()

### What this means

* **UMAP**: Cell types separate cleanly. Tumor takes up a large cluster in the bottom-center/right, with fibroblast sitting next to it. Intestinal epithelial occupies the left side. The high-count/high-mt UMAP clusters from earlier correspond to tumor and epithelial compartments, which confirms that the elevated mt% there is biological.

* **Spatial**: The cell types land in anatomically sensible regions. Tumor forms a contiguous mass. Fibroblasts wrap around it as stroma. Intestinal epithelium lines the mucosal surface, and immune cells are interspersed throughout.

## Our Marker-Based Annotations

Now we annotate bins ourselves using canonical CRC marker gene sets. It is a common approach when ground-truth labels are not available.

**How `sc.tl.score_genes` works:** For each marker panel, scanpy computes a per-bin enrichment score. It takes the mean expression of the marker genes, subtracts the mean expression of a reference set of randomly selected genes with similar expression levels, and returns the difference. That controls for overall expression level, so a bin is not labeled "Tumor" just because it has a lot of total counts. Each bin then gets assigned to whichever panel scored highest. Bins where the top score is too low (<0.05) or too close to the runner-up (margin <0.02) get labeled "Low_confidence."

We use the same broad categories as the ground truth (UnsupervisedL1) so the two annotation sets line up side by side.

In [ ]:
%%time
# Marker-based annotation using broad cell types matching ground truth
print("Scoring marker panels...")

marker_sets = {
    "Tumor":                  ["EPCAM", "KRT8", "KRT18", "KRT19", "KRT20", "CEACAM5", "MUC1"],
    "Fibroblast":             ["COL1A1", "COL1A2", "COL3A1", "DCN", "LUM", "COL6A1"],
    "Endothelial":            ["PECAM1", "VWF", "EMCN", "KDR", "RAMP2", "PLVAP"],
    "Myeloid":                ["LST1", "TYROBP", "FCER1G", "CTSS", "AIF1", "C1QC", "C1QB"],
    "T cells":                ["CD3D", "CD3E", "TRBC1", "TRBC2", "NKG7", "IL7R", "LTB"],
    "B cells":                ["MS4A1", "CD79A", "CD79B", "MZB1", "JCHAIN", "IGKC", "CD74"],
    "Smooth Muscle":          ["ACTA2", "MYH11", "DES", "TAGLN", "CNN1", "ACTG2"],
    "Intestinal Epithelial":  ["MUC2", "TFF3", "FCGBP", "CLCA1", "ZG16", "AGR2",
                               "FABP1", "FABP2", "ALPI", "VIL1", "SIS"],
}

present_markers = {}
for ct, genes in marker_sets.items():
    present = [g for g in genes if g in adata_8um.var_names]
    present_markers[ct] = present
    print(f"  {ct:25s}  {len(present):>2}/{len(genes)} genes  {present}")

score_cols = []
for ct, genes in present_markers.items():
    if len(genes) >= 2:
        score_col = f"{ct}_score"
        sc.tl.score_genes(adata_8um, gene_list=genes, score_name=score_col, use_raw=False)
        score_cols.append(score_col)

score_df = adata_8um.obs[score_cols].copy()
adata_8um.obs["cell_type_marker"] = score_df.idxmax(axis=1).str.replace("_score", "", regex=False)

max_score = score_df.max(axis=1)
sorted_scores = np.sort(score_df.values, axis=1)
second_score = sorted_scores[:, -2] if sorted_scores.shape[1] > 1 else np.zeros(len(score_df))
margin = max_score.values - second_score
adata_8um.obs.loc[(max_score < 0.05) | (margin < 0.02), "cell_type_marker"] = "Low_confidence"

print("\nMarker-based cell type distribution:")
print(adata_8um.obs["cell_type_marker"].value_counts().to_string())

### What this means

* Every bin gets scored against every marker panel and assigned to the highest scorer. The distribution below shows what came out.

* **Why so many fibroblasts?** Fibroblast markers (COL1A1, DCN, LUM) are among the most broadly expressed genes in solid tissue. They are not specific to fibroblasts. Cancer-associated fibroblasts, myofibroblasts, some smooth muscle cells, and even some tumor cells express the same collagens. At 8 um resolution where one bin can overlap several cells, a bin at the tumor-stroma interface may pick up both tumor and stromal transcripts, and whichever panel wins by a hair gets the label. This is a known limitation of marker-based scoring on spatial data.

In [ ]:
# Spatial maps of our marker-based annotations
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
ct_col = "cell_type_marker"
cell_types = sorted(adata_8um.obs[ct_col].unique())
palette_m = cell_type_palette  # shared with the ground-truth plot above

for ct in cell_types:
    mask = (adata_8um.obs[ct_col] == ct).values
    axes[0].scatter(adata_8um.obsm["X_umap"][mask, 0], adata_8um.obsm["X_umap"][mask, 1],
                    s=0.5, label=ct, alpha=0.5, rasterized=True, color=palette_m[ct])
axes[0].legend(markerscale=8, fontsize=7, loc="best"); axes[0].set_title("UMAP: Our annotations")

coords = adata_8um.obsm["spatial"]
for ct in cell_types:
    mask = (adata_8um.obs[ct_col] == ct).values
    axes[1].scatter(coords[mask, 0], coords[mask, 1],
                    s=0.3, label=ct, alpha=0.4, rasterized=True, color=palette_m[ct])
axes[1].legend(markerscale=10, fontsize=7, loc="best")
axes[1].set_title("Spatial: Our annotations"); axes[1].set_aspect("equal"); axes[1].invert_yaxis(); axes[1].axis("off")
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_marker_cell_type_maps.png", dpi=150, bbox_inches="tight")
plt.show()

### What this means

* The spatial patterns hold up well. Tumor fills the lower-right, fibroblast fills the stroma, and intestinal epithelium lines the mucosal surface. The spatial architecture is consistent between our annotations and the ground truth, even in places where the specific labels disagree.

* As in any scRNA-seq workflow, you can sanity-check the assignments by plotting the canonical markers per assigned cell type on a dotplot (`sc.pl.dotplot`). High mean expression and a high fraction of expressing bins along the diagonal tell you the marker panels are catching the right populations. Off-diagonal signal flags non-specific markers (the fibroblast collagen panel here is a known offender).

* We would also quantify this against the ground-truth labels, using a per-cell-type median marker-score heatmap plus a normalized confusion matrix versus `UnsupervisedL1`, to put a number on the agreement and flag which panels (the non-specific fibroblast collagens here) drive the disagreement; omitted to keep the focus on the spatial result.

## Neighborhood Analysis

Which cell types sit next to each other in tissue? **Neighborhood enrichment** answers that by comparing the actual frequency of cell type pairs among spatial neighbors to the frequency you would expect from a random permutation. If Tumor bins are surrounded by other Tumor bins more often than chance, that shows up as self-enrichment. If T cells steer clear of the tumor region, it shows up as negative enrichment.

The algorithm:
1. Build a spatial neighbor graph (k=20 nearest bins)
2. Count how often each pair of cell types appears as neighbors
3. Randomly shuffle the cell type labels many times and recount
4. Compare the real counts to the permuted distribution (z-score)

In [ ]:
%%time
# Neighborhood enrichment
ct_key = "UnsupervisedL1"
adata_nhood = adata_8um[adata_8um.obs[ct_key].notna()].copy()
adata_nhood.obs[ct_key] = adata_nhood.obs[ct_key].astype("category")

sq.gr.spatial_neighbors(adata_nhood, n_neighs=20, coord_type="generic")
sq.gr.nhood_enrichment(adata_nhood, cluster_key=ct_key)

fig, ax = plt.subplots(figsize=(8, 7))
sq.pl.nhood_enrichment(adata_nhood, cluster_key=ct_key, ax=ax)
ax.set_title("Neighborhood Enrichment (ground truth labels)")
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_nhood_enrichment.png", dpi=150, bbox_inches="tight")
plt.show()

### What this means

* **Strong self-enrichment** (bright diagonal) for Tumor, Smooth Muscle, and Intestinal Epithelial. Each of these types forms a contiguous spatial domain.

* **Tumor avoids most other types**: dark entries fill the Tumor row and column off the diagonal. This is consistent with a dense tumor mass and limited immune infiltration. The Tumor-T cell avoidance in particular suggests immune exclusion, which matters for the immunotherapy context.

* **Fibroblast-Intestinal Epithelial**: Negative enrichment, even though both types are abundant. They occupy adjacent but distinct spatial compartments.

## Co-occurrence

Neighborhood enrichment tells you whether two cell types touch as direct neighbors more often than chance. **Co-occurrence** asks the next question: how does that probability change as you move outward from an anchor cell? It computes P(other | anchor) at increasing radii, normalized so 1.0 means independence. Values above 1 mean co-localization at that scale, below 1 means avoidance, and the curves usually flatten back to 1 once you are far enough out that the types become independent.

Why bother if you already have neighborhood enrichment? Because a lot of biology happens at distance, not at direct contact. In tumor immunology, T cells are often excluded from direct contact with tumor cells but pile up at the invasive margin some tens to hundreds of microns out. Neighborhood enrichment alone would just say "T cells avoid tumor." Co-occurrence shows the curve dipping below 1 at short range and rising at intermediate range, which is the spatial signature of an excluded but recruited population (and a real prognostic feature in colorectal cancer). The same idea covers paracrine signaling, tertiary lymphoid structures next to tumors, and stem cell niches where the supporting cells sit a few cell diameters away rather than touching.

In [ ]:
%%time
# Co-occurrence: how does co-localization change with distance?
try:
    sq.gr.co_occurrence(adata_nhood, cluster_key=ct_key)

    co = adata_nhood.uns[f"{ct_key}_co_occurrence"]
    occ = co["occ"]
    interval = co["interval"]

    if len(interval) == occ.shape[-1] + 1:
        x = (interval[:-1] + interval[1:]) / 2
    else:
        x = interval

    cats = list(adata_nhood.obs[ct_key].cat.categories)
    n = len(cats)
    ncols = 5
    nrows = int(np.ceil(n / ncols))
    palette_co = dict(zip(cats, plt.cm.tab10(np.linspace(0, 1, n))))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows),
                             sharex=True, sharey=True)
    axes_flat = axes.flat if hasattr(axes, "flat") else [axes]

    for i, anchor in enumerate(cats):
        ax = axes_flat[i]
        for j, other in enumerate(cats):
            ax.plot(x, occ[i, j, :], label=other, color=palette_co[other], lw=1.2)
        ax.axhline(1.0, color="gray", ls="--", lw=0.7)
        ax.set_title(f"Anchor: {anchor}", fontsize=9)
        ax.set_xlabel("Distance"); ax.set_ylabel("P(other | anchor)")

    for k in range(n, len(list(axes_flat))):
        axes_flat[k].axis("off")

    handles, labels = axes_flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="center right", fontsize=8,
               bbox_to_anchor=(1.02, 0.5), title=ct_key)
    plt.suptitle("Co-occurrence by cell type", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "crc_co_occurrence.png", dpi=150, bbox_inches="tight")
    plt.show()
except Exception as e:
    print(f"Co-occurrence analysis: {e}")

del adata_nhood; gc.collect()

### What this means

* **Co-occurrence** shows how the probability of finding each cell type changes with distance from an anchor type. Values >1 mean co-localization, <1 means avoidance, and the curve decays toward 1 at larger distances.

* **Tumor**: Self-enriched at short range; every other type sits below 1. Reinforces the immune exclusion pattern.

* **Neuronal**: Massively self-enriched at short range, since enteric neurons cluster in the myenteric plexus. Smooth Muscle comes in as a secondary neighbor.

* **T cells**: Moderate self-enrichment, with a slight bump for B cell and Myeloid co-occurrence at short range. Consistent with immune niches in the stroma.

* The difference from neighborhood enrichment: co-occurrence tells you how the relationship changes with distance. Two cell types can co-localize as immediate neighbors and still avoid each other at longer ranges.

## Spatially Variable Genes (SVGs)

Standard differential expression (DE) asks: which genes differ between groups (Tumor vs. Fibroblast, say)? SVG detection asks a different question: which genes have **spatially structured expression patterns**, regardless of any grouping?

**Moran's I** measures spatial autocorrelation. For each gene, it compares expression at each bin to expression at neighboring bins. If nearby bins tend to share expression levels, the gene gets a high I value. If expression is random with respect to location, I stays near zero.

This is useful because:
- It surfaces spatially patterned genes without requiring prior cell type labels
- It can pick up genes with gradients, boundary effects, or regional enrichment that DE would miss
- It validates whether known markers actually are spatially structured in this tissue

In [ ]:
%%time
# SVG detection (subsampled for speed)
n_svg = min(10_000, adata_8um.shape[0])
print(f"Subsampling to {n_svg:,} bins for SVG analysis...")
adata_svg = sc.pp.subsample(adata_8um, n_obs=n_svg, copy=True)
sq.gr.spatial_neighbors(adata_svg, n_neighs=20, coord_type="generic")

if "highly_variable" in adata_svg.var.columns:
    adata_svg_hvg = adata_svg[:, adata_svg.var["highly_variable"]].copy()
else:
    adata_svg_hvg = adata_svg.copy()

print(f"Running Moran's I on {adata_svg_hvg.shape[1]} genes...")
sq.gr.spatial_autocorr(adata_svg_hvg, mode="moran", n_jobs=1)

morans = adata_svg_hvg.uns["moranI"].sort_values("I", ascending=False)
print(f"\nTop 20 SVGs:")
print(morans.head(20)[["I", "pval_norm"]].to_string())

# Plot specific genes from different tissue compartments
# These are the genes we interpret below, so we plot exactly these
plot_genes = ["PIGR", "MUC2", "MUC12", "FCGBP", "DES", "MYH11", "COL1A1", "COL3A1", "CEACAM5"]

# Check which are present in the data
available = [g for g in plot_genes if g in adata_svg.var_names]
missing = [g for g in plot_genes if g not in adata_svg.var_names]
if missing:
    print(f"\nGenes not in data (skipping): {missing}")
print(f"Plotting {len(available)} genes: {available}")

n_genes = len(available)
n_cols = min(5, n_genes)
n_rows = (n_genes + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
if n_rows == 1:
    axes = np.atleast_2d(axes)
coords = adata_svg.obsm["spatial"]

for idx, gene in enumerate(available):
    r, c = divmod(idx, n_cols)
    ax = axes[r, c]
    vals = adata_svg[:, gene].X
    expr = vals.toarray().flatten() if sparse.issparse(vals) else np.asarray(vals).flatten()
    i_val = morans.loc[gene, "I"] if gene in morans.index else "N/A"
    i_str = f"{i_val:.3f}" if isinstance(i_val, float) else i_val
    sc_p = ax.scatter(coords[:, 0], coords[:, 1], c=expr, s=1, cmap="magma",
                      edgecolors="none", rasterized=True)
    ax.set_title(f"{gene} (I={i_str})", fontsize=11)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
    plt.colorbar(sc_p, ax=ax, shrink=0.6)

# Turn off unused axes
for idx in range(n_genes, n_rows * n_cols):
    r, c = divmod(idx, n_cols)
    axes[r, c].axis("off")

plt.suptitle("Spatially Variable Genes", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_svgs.png", dpi=150, bbox_inches="tight")
plt.show()

del adata_svg, adata_svg_hvg; gc.collect()

### What this means

* The plotted genes line up with different tissue compartments. **PIGR/MUC2/MUC12/FCGBP** are mucosal epithelium markers concentrated in the upper-left glandular region. **DES/MYH11** are smooth muscle, localized to the muscularis layer. **COL1A1/COL3A1** are fibroblast/stromal genes. **CEACAM5** is a tumor epithelial marker concentrated in the tumor mass.

* Signal looks sparse (most bins are dark) because dropout is high at 8 um depth. Moran's I still captures the spatial coherence of the non-zero bins, even under heavy dropout.

* All p-values come out as 0 with 10K bins, so what matters for ranking is the I value itself, not statistical significance.

## Cell-Cell Communication

Cells signal to each other through ligand-receptor (L-R) pairs. In spatial data an L-R interaction is only believable if the ligand-expressing and receptor-expressing cell types sit close together in the tissue.

We use **LIANA**, which runs five published L-R methods (CellPhoneDB, CellChat, NATMI, Connectome, SingleCellSignalR) and combines them into one consensus rank. This is the tool the source paper ([Oliveira et al., Nat Genet 2025](https://www.nature.com/articles/s41588-025-02193-3)) used for its Figure 5 analysis.

LIANA is slow on a tissue this size and its output is a small table, so we ran it once offline on the full ~500K-bin P1 tissue and ship the result as `crc_p1_liana_full.parquet`. The cell below loads that file and draws two dotplots that match the paper: the left one is the ECM story (macrophage subtypes plus CAFs to tumor cells, Fig 5C), the right one is the chemokine story (macrophage subtypes to CD4/CD8 T cells, Fig 5D).

In [ ]:
# Load precomputed LIANA results + render two paper-aligned dotplots
import pyarrow.parquet as pq

if not LIANA_PARQUET.exists():
    raise FileNotFoundError(
        f"Missing {LIANA_PARQUET.name}. Re-run the install cell (it pulls "
        "from Figshare), or rerun scripts/generate_precomputed.py locally."
    )

table = pq.read_table(LIANA_PARQUET)
liana_df = table.to_pandas()
meta = dict(table.schema.metadata or {})
provenance = {
    k.decode().replace("liana_", ""): v.decode(errors="replace")
    for k, v in meta.items()
    if k.decode().startswith("liana_") and k.decode() != "liana_mac_subtype_table_parquet"
}
print(f"LIANA rows: {len(liana_df):,}")
for k in ("groupby", "resource", "n_perms", "n_bins", "subset", "dataset"):
    if k in provenance:
        print(f"  {k+':':10s} {provenance[k]}")


def _liana_dotplot(ax, df, senders, receivers,
                   top_per_pair=3, top_lr=15, title=""):
    """Render one paper-style L-R dotplot for (senders -> receivers) on `ax`."""
    sub = df[df["source"].isin(senders) & df["target"].isin(receivers)].copy()
    if sub.empty:
        ax.set_title(f"{title}\n(no rows for these sources/targets)", fontsize=10)
        ax.axis("off")
        return

    sub["lr"] = sub["ligand_complex"] + " > " + sub["receptor_complex"]
    sub["pair"] = sub["source"].astype(str) + " > " + sub["target"].astype(str)
    tp = (sub.sort_values("magnitude_rank")
            .groupby(["source", "target"], observed=True).head(top_per_pair))
    lr_pairs = tp["lr"].value_counts().head(top_lr).index.tolist()
    sub = sub[sub["lr"].isin(lr_pairs)]

    pair_order = [f"{s} > {t}" for s in senders for t in receivers
                  if f"{s} > {t}" in sub["pair"].unique()]
    lr_score = (sub.assign(s=lambda d: -np.log10(d["magnitude_rank"].clip(lower=1e-4)))
                  .groupby("lr")["s"].sum().sort_values(ascending=False))
    lr_order = lr_score.index.tolist()

    mag = sub.pivot(index="lr", columns="pair",
                    values="magnitude_rank").reindex(index=lr_order, columns=pair_order)
    mean_v = sub.pivot(index="lr", columns="pair",
                       values="lr_means").reindex(index=lr_order, columns=pair_order)

    for i in range(len(lr_order)):
        for j in range(len(pair_order)):
            m, v = mag.iloc[i, j], mean_v.iloc[i, j]
            if pd.isna(m) or pd.isna(v):
                continue
            sig = -np.log10(max(m, 1e-4))
            size = min(20 + v * 60, 280)
            ax.scatter(j, i, s=size, c=sig, cmap="viridis", vmin=0, vmax=3,
                       edgecolors="black", linewidths=0.3, alpha=0.9)

    ax.set_yticks(range(len(lr_order))); ax.set_yticklabels(lr_order, fontsize=8)
    ax.set_xticks(range(len(pair_order)))
    ax.set_xticklabels(pair_order, fontsize=8, rotation=45, ha="right")
    ax.set_xlim(-0.5, len(pair_order) - 0.5)
    ax.set_ylim(-0.5, len(lr_order) - 0.5)
    ax.invert_yaxis()
    ax.set_title(title, fontsize=11)
    ax.grid(alpha=0.2, linewidth=0.5)


# -- Two paper-aligned dotplots: ECM story (Fig 5C) + chemokine/T-cell story (Fig 5D) --
ecm_senders = ["Macrophage_SPP1+", "Macrophage_SELENOP+", "Macrophage_other",
               "CAF", "Fibroblast"]
ecm_receivers = ["Tumor II", "Tumor III", "Tumor V"]

chemo_senders = ["Macrophage_SPP1+", "Macrophage_SELENOP+", "Macrophage_other"]
chemo_receivers = ["CD4 T cell", "CD8 T cell"]

fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(18, 7))
_liana_dotplot(ax_left, liana_df, ecm_senders, ecm_receivers,
               title="ECM signaling (Oliveira Fig 5C):\nmacrophage subtypes + CAFs -> Tumor II/III/V")
_liana_dotplot(ax_right, liana_df, chemo_senders, chemo_receivers,
               title="Chemokine signaling (Oliveira Fig 5D):\nmacrophage subtypes -> CD4/CD8 T cells")

sm = plt.cm.ScalarMappable(norm=plt.Normalize(0, 3), cmap="viridis"); sm.set_array([])
fig.colorbar(sm, ax=[ax_left, ax_right], shrink=0.6, pad=0.02
             ).set_label("-log10(magnitude_rank)", fontsize=9)
fig.suptitle("LIANA top L-R edges  (size = mean expression, color = significance)",
             fontsize=12, y=1.02)
plt.savefig(FIG_DIR / "crc_liana_dotplot.png", dpi=150, bbox_inches="tight")
plt.show()

gc.collect()

### What this means

* **Left dotplot, ECM signaling (paper Fig 5C)**: senders are macrophage subtypes plus CAFs/fibroblasts, receivers are Tumor II/III/V. The strongest edges from `Macrophage_SPP1+` are collagen and VIM ligands hitting `CD44` and `DDR1` (`COL1A1`/`COL3A1`-`CD44`, `COL3A1`-`DDR1`, `VIM`-`CD44`). CAFs drive the same edges, usually with the largest single magnitude (`CAF` to `Tumor II`, `COL1A1`-`CD44`). These are the pro-tumor ECM-remodeling ligands the paper assigns to SPP1+ macrophages and CAFs.

* **Right dotplot, chemokine signaling (paper Fig 5D)**: senders are macrophage subtypes, receivers are CD4/CD8 T cells, so the chemokine edges are not buried under collagen rows. The paper's SELENOP+ finding is T-cell recruitment via `CXCL9`/`CXCL10`/`CXCL11` to `CXCR3`. Look in the `Macrophage_SELENOP+` to `CD8 T cell` (and CD4) column for these pairs. Whether they appear depends on how many SELENOP+ bins survive 8 um depth, so read the dot color and size in that column for this run.

* **Scale**: the parquet was computed on the full ~500K-bin tissue (`subset: full_tissue` in the provenance print), so these L-R statistics are whole-tissue, not from the 50K subsample used elsewhere.

## Beyond This Tutorial

This notebook covers a core spatial transcriptomics workflow, but a number of other analyses are commonly run on Visium HD data:

### Deconvolution

At 8 um resolution, a single bin may pick up signal from 1 to 3 cells. Deconvolution methods (cell2location, RCTD, stereoscope) estimate the fractional composition of cell types within each bin using a single-cell RNA-seq reference. They become especially valuable at coarser resolutions (16 um, 55 um Visium v1) where bins average over many cells.

### Spatial domains / compartment detection

SpaGCN, STAGATE, BayesSpace, and GraphST identify spatially coherent regions (domains) that share similar expression profiles. Rather than clustering bins in gene-expression space alone, these methods incorporate spatial neighbor graphs to find contiguous tissue compartments such as tumor core, invasive margin, or immune-rich stroma. People sometimes call this "spatial clustering" or "tissue compartment finding."

### Other spatial analyses

- **Spatial trajectory analysis** (stLearn, PAGA with spatial constraints): traces differentiation or disease progression paths through tissue space
- **Niche analysis**: characterize the cellular microenvironment around each cell type and identify distinct tissue niches
- **Gene regulatory network inference**: methods like SCENIC+ adapted for spatial data, to find spatially regulated transcription factor programs
- **Multi-sample integration**: tools like PASTE, SLAT, or STAligner for aligning and comparing spatial data across multiple tissue sections or patients

## Summary

1. **QC**: We dropped ~22% of bins using per-bin thresholds (UMIs, genes, mt%) and spatial outlier detection.
2. **Ground truth**: The original study's annotations show well-separated cell types with anatomically sensible spatial patterns.
3. **Our annotations**: Marker-based scoring on broad cell types that match the ground-truth categories. The spatial architecture reproduces the ground truth, though non-specific fibroblast markers and immune gene dropout at shallow depth blur the per-bin labels.
4. **Spatial analysis**: Neighborhood enrichment and co-occurrence both confirm tumor self-clustering with immune exclusion. SVGs line up with expected tissue compartment markers.
5. **Cell-cell communication**: LIANA picks out CAFs and macrophages as the dominant senders of ECM ligands (collagen, VIM) to tumor cells via CD44/DDR1/integrin receptors. That is the same pro-tumor TME signaling axis the source paper describes.